In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from adjustText import adjust_text

def load_and_report_losses(csv_path):
    df = pd.read_csv(csv_path)

    # Ensure numeric
    df['losses'] = pd.to_numeric(df.get('losses', 0), errors='coerce').fillna(0)
    df['capacity'] = pd.to_numeric(df.get('capacity', 0), errors='coerce').fillna(0)

    # Compute loss percentage per segment safely
    df['loss_percent'] = df.apply(
        lambda row: (row['losses'] / row['capacity'] * 100) if row['capacity'] > 0 else np.nan,
        axis=1
    )

    # Filter only built & non-zero-loss segments
    df_losses = df[(df['capacity'] > 0) & (df['losses'] > 0)].copy()

    # Report table as before...
    df_report = df_losses[['id', 'from_node', 'to_node', 'length',
                           'hp_type', 'capacity', 'costs', 'losses', 'loss_percent']]
    df_report = df_report.sort_values('from_node', ascending=True)

    print("=== Pipe Segment Heat Loss Report ===")
    print(df_report.to_string(index=False))
    print("-------------------------------------")
    total_loss = df_losses['losses'].sum()
    mean_loss_percent = df_losses['loss_percent'].mean()
    print(f"Total heat loss (sum over segments)     : {total_loss:.2f} kW")
    print(f"Average loss percent (per segment)      : {mean_loss_percent:.2f}%")

    return df_losses, df_report

def load_and_plot_losses(csv_path):
    df = pd.read_csv(csv_path)
    df['losses'] = pd.to_numeric(df.get('losses', 0), errors='coerce').fillna(0)
    df['capacity'] = pd.to_numeric(df.get('capacity', 0), errors='coerce').fillna(0)

    df['loss_percent'] = df.apply(
        lambda r: (r['losses'] / r['capacity'] * 100) if r['capacity'] > 0 else np.nan,
        axis=1
    )

    df_losses = df[(df['capacity'] > 0) & (df['losses'] > 0)].copy()

    fig, ax = plt.subplots(figsize=(12, 10))
    sc = ax.scatter(df_losses['length'], df_losses['losses'],
                    c=df_losses['loss_percent'], cmap='viridis', edgecolor='k', s=100, alpha=0.7 )

    # Add colorbar
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('Loss percent (%)', fontsize=16)
    # Annotate points
    texts = []
    for _, row in df_losses.iterrows():
        label = f"{row['from_node']} → {row['to_node']}\nCap: {row['capacity']:,.2f} kW"
        texts.append(
            ax.text(row['length'], row['losses'], label, fontsize=10, fontstyle='normal',
                    path_effects=[pe.withStroke(linewidth=3, foreground="white")])
        )

    adjust_text(texts, arrowprops=dict(arrowstyle="-", color='gray'))

    ax.set_xlabel('Pipe length (m)', fontsize=16)
    ax.set_ylabel('Heat losses (kW)', fontsize=16)
    ax.tick_params(axis='both', labelsize=16)
    ax.set_title('Pipe segment length vs losses')
    ax.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    csv_path = "../3_outputs/DHNx_step1_result.csv"  # or your actual file
    df_losses, df_report = load_and_report_losses(csv_path)
    load_and_plot_losses(csv_path)
